In [1]:
import os
import glob
import random
from collections import Counter

# Settings
SEED = 42
SAMPLE_SIZE = 6000

# Repo root (so paths behave even if the notebook runs from /analysis)
REPO_ROOT = "/Users/radinabakalov/TECHTRACK-RADINABAKALOV"

# Where all the raw logistics images + YOLO label files live
DATA_DIR = "/Users/radinabakalov/TECHTRACK-RADINABAKALOV/techtrack/storage/logistics"

# We'll write out the final list of sampled image paths here
OUT_LIST_PATH = os.path.join(REPO_ROOT, "analysis", "sample_images.txt")


def read_classes_from_label(txt_path: str) -> set[int]:
    """
    Open a YOLO-format .txt label file and pull out
    which object classes appear in it (by their integer id).
    Returns an empty set if the file doesn't exist.
    """
    classes = set()

    # No label file means no annotations (just skip it)
    if not os.path.exists(txt_path):
        return classes

    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            # First token on each line is the class id in YOLO format
            classes.add(int(parts[0]))

    return classes


def get_label_path(img_path: str) -> str:
    """Helper: swap .jpg -> .txt to find the matching label file."""
    base, _ = os.path.splitext(img_path)
    return base + ".txt"

# 1) Look at all images in the dataset folder 
all_images = sorted(glob.glob(os.path.join(DATA_DIR, "*.jpg")))
if not all_images:
    raise FileNotFoundError(f"No .jpg files found in: {DATA_DIR}")

print("Total images found:", len(all_images))


# 2) Draw a reproducible random sample
# Fix the seed so we get the same sample every time we run this
random.seed(SEED)

# If there are fewer images than SAMPLE_SIZE, just take everything
sampled_images = random.sample(all_images, k=min(SAMPLE_SIZE, len(all_images)))
print("Initial sample size:", len(sampled_images))


# 3) Check class coverage 
# We want to make sure every class that exists in the full dataset
# is represented at least once in our sample. 
# Otherwise the sample would be blind to some object types, which messes up analysis.
def class_presence(image_paths: list[str]) -> Counter:
    """Count how many images contain each class (at least once per image)."""
    counts = Counter()
    for img_path in image_paths:
        label_path = get_label_path(img_path)
        for c in read_classes_from_label(label_path):
            counts[c] += 1
    return counts

# Tally up class presence across the full dataset vs. our sample
full_presence = class_presence(all_images)
sample_presence = class_presence(sampled_images)

full_classes = set(full_presence.keys())
sample_classes = set(sample_presence.keys())
missing_classes = sorted(full_classes - sample_classes)

print("Classes in full dataset:", len(full_classes))
print("Classes in sample:", len(sample_classes))
print("Missing classes in sample:", missing_classes)

# 4) Patch in any missing classes 
# Rare classes might not have been caught by our random sample,
# so we scan through the dataset and grab extra images to fill the gaps.
if missing_classes:
    needed = set(missing_classes)
    extras = []

    for img_path in all_images:
        # Once we've covered every missing class, we can stop looking
        if not needed:
            break
        label_path = get_label_path(img_path)
        present = read_classes_from_label(label_path)

        # Does this image happen to contain a class we're still missing?
        hit = needed.intersection(present)
        if hit:
            extras.append(img_path)
            # Cross those classes off the "still needed" list
            needed -= hit

    # Merge the extras into our sample and de-duplicate just in case
    sampled_images.extend(extras)
    sampled_images = sorted(set(sampled_images)) 

    print("Added extra images for coverage:", len(extras))
    print("New sample size:", len(sampled_images))

# 5) Save the final sample list to disk 
# This lets other scripts re-use the exact same sample without re-running this one.
os.makedirs(os.path.dirname(OUT_LIST_PATH), exist_ok=True)

with open(OUT_LIST_PATH, "w") as f:
    for path in sampled_images:
        f.write(path + "\n")

print("Saved sample list to:", OUT_LIST_PATH)


Total images found: 9525
Initial sample size: 6000
Classes in full dataset: 20
Classes in sample: 20
Missing classes in sample: []
Saved sample list to: /Users/radinabakalov/TECHTRACK-RADINABAKALOV/analysis/sample_images.txt
